# CNN Training — Intel Image Classification

Training 16 arsitektur Keras Conv2D, evaluasi macro F1, lalu validasi dengan implementasi from-scratch NumPy.

**Struktur eksperimen:**
1. Training 16 arsitektur (2×2×2×2 grid)
2. Analisis per-hyperparameter
3. Keras vs From-Scratch (arsitektur terbaik)
4. Conv2D vs LocallyConnected2D

In [ ]:
import os, sys, itertools, time, json, glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import f1_score, classification_report

# Cari project root (parent dari src/)
root = Path(os.getcwd())
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
os.chdir(root)
print('Root:', root)
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
# Sesuaikan path ini kalau struktur dataset-nya beda
DATA_DIR   = Path('data/intel')          # data/intel/train/{class}/*.jpg
MODELS_DIR = Path('models/cnn')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE   = 150
BATCH_SIZE = 64
EPOCHS     = 20
NUM_CLASSES = 6

# Kaggle path override (uncomment kalau running di Kaggle)
# DATA_DIR = Path('/kaggle/input/intel-image-classification')
# TRAIN_DIR = DATA_DIR / 'seg_train/seg_train'
# TEST_DIR  = DATA_DIR / 'seg_test/seg_test'

TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR  = DATA_DIR / 'test'

## 1. Load Dataset

In [ ]:
CLASSES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
print('Classes:', CLASSES)

# Intel Image Classification tidak menyediakan folder val terpisah.
# Solusi: split 80/20 dari TRAIN_DIR menggunakan validation_split Keras.
# Test set (TEST_DIR) tidak disentuh selama training -> tidak ada data leakage.
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, label_mode='int', seed=42, class_names=CLASSES,
    validation_split=0.2, subset='training',
)
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, label_mode='int', seed=42, class_names=CLASSES,
    validation_split=0.2, subset='validation',
)
test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, label_mode='int', shuffle=False, class_names=CLASSES,
)

# Normalize ke [0, 1]
norm = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds_raw.map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds_raw.map(lambda x,y:   (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds_raw.map(lambda x,y:  (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)

# True labels test set untuk F1
y_test = np.concatenate([y.numpy() for _, y in test_ds])
print(f'Train batches: {len(train_ds)}, Val batches: {len(val_ds)}, Test samples: {len(y_test)}')


## 2. Definisi 16 Arsitektur CNN

In [ ]:
# 2^4 = 16 kombinasi
GRID = {
    'n_conv':  [2, 4],
    'filters': [[32, 64], [64, 128]],
    'k_size':  [3, 5],
    'pool':    ['max', 'avg'],
}

configs = list(itertools.product(
    GRID['n_conv'], GRID['filters'], GRID['k_size'], GRID['pool']
))
print(f'Total configs: {len(configs)}')
for i, (n, f, k, p) in enumerate(configs):
    print(f'  [{i:02d}] n_conv={n}, filters={f}, k_size={k}, pool={p}')

In [ ]:
def build_cnn(n_conv, filters, k_size, pool_type):
    """Keras Sequential CNN dengan Conv2D blocks."""
    PoolLayer = (tf.keras.layers.MaxPooling2D if pool_type == 'max'
                 else tf.keras.layers.AveragePooling2D)

    model = tf.keras.Sequential(name=f'cnn_c{n_conv}_k{k_size}_{pool_type}')
    model.add(tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)))

    for i in range(n_conv):
        f = filters[i % len(filters)]
        model.add(tf.keras.layers.Conv2D(f, k_size, activation='relu', padding='same'))
        model.add(PoolLayer(2, 2))

    model.add(tf.keras.layers.GlobalAveragePooling2D())
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'))

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Quick sanity check
tmp = build_cnn(2, [32,64], 3, 'max')
tmp.summary()

## 3. Training 16 Arsitektur

In [ ]:
# Load histories yang sudah ada dari cache
HIST_PATH = MODELS_DIR / 'histories.json'
histories = {}
if HIST_PATH.exists():
    with open(HIST_PATH) as fp:
        histories = json.load(fp)

all_results = []
early_stop  = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

for i, (n_conv, filters, k_size, pool) in enumerate(configs):
    tag       = f'c{n_conv}_f{"_".join(map(str,filters))}_k{k_size}_{pool}'
    save_path = MODELS_DIR / f'arch_{i:02d}_{tag}.h5'

    print(f'[{i+1}/16] {tag}', end=' ... ')

    if save_path.exists():
        model = tf.keras.models.load_model(save_path)
        print('loaded from cache')
    else:
        model = build_cnn(n_conv, filters, k_size, pool)
        hist  = model.fit(
            train_ds, epochs=EPOCHS,
            validation_data=val_ds,   # val_ds, bukan test_ds
            callbacks=[early_stop], verbose=0,
        )
        model.save(save_path)
        histories[tag] = hist.history
        # Simpan histories setiap kali model baru selesai
        with open(HIST_PATH, 'w') as fp:
            json.dump(histories, fp)
        print(f'trained {len(hist.history["loss"])} epochs')

    y_pred = model.predict(test_ds, verbose=0).argmax(1)
    f1     = f1_score(y_test, y_pred, average='macro')

    all_results.append({
        'idx': i, 'tag': tag,
        'n_conv': n_conv, 'filters': str(filters),
        'k_size': k_size, 'pool': pool,
        'f1_macro': round(f1, 4),
        'params': model.count_params(),
    })
    print(f'  -> F1: {f1:.4f}')

# Simpan hasil
with open(MODELS_DIR / 'results.json', 'w') as fp:
    json.dump(all_results, fp, indent=2)
print('\nDone!')


## 4. Analisis Hasil Arsitektur

In [ ]:
df = pd.DataFrame(all_results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(df[['tag','n_conv','filters','k_size','pool','f1_macro','params']])

best_row = df.iloc[0]
print(f'\nArsitektur terbaik: {best_row["tag"]} (F1={best_row["f1_macro"]:.4f})')

In [ ]:
# Rata-rata F1 per nilai hyperparameter
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
hparams   = [('n_conv','Jumlah Conv Layer'), ('filters','Jumlah Filter'),
             ('k_size','Ukuran Kernel'),      ('pool','Jenis Pooling')]

for ax, (col, title) in zip(axes.flat, hparams):
    grp = df.groupby(col)['f1_macro'].mean().sort_values(ascending=False)
    ax.bar(grp.index.astype(str), grp.values, color='steelblue')
    ax.set_title(title)
    ax.set_ylabel('Mean F1-macro')
    ax.set_ylim(max(0, grp.min() - 0.05), grp.max() + 0.05)
    for j, (x, v) in enumerate(zip(grp.index.astype(str), grp.values)):
        ax.text(j, v + 0.002, f'{v:.4f}', ha='center', fontsize=9)

plt.suptitle('Pengaruh Tiap Hyperparameter terhadap Macro F1', y=1.02)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'hparam_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Load histories dari file JSON jika ada ---
HIST_PATH = MODELS_DIR / 'histories.json'
if HIST_PATH.exists():
    with open(HIST_PATH) as fp:
        histories.update(json.load(fp))

# 1. Loss curve arsitektur terbaik
best_tag = best_row['tag']
if best_tag in histories:
    h = histories[best_tag]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(h['loss'], label='train'); ax1.plot(h['val_loss'], label='val')
    ax1.set_title('Loss'); ax1.legend()
    ax2.plot(h['accuracy'], label='train'); ax2.plot(h['val_accuracy'], label='val')
    ax2.set_title('Accuracy'); ax2.legend()
    plt.suptitle(f'Training Curves - Arsitektur Terbaik ({best_tag})')
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'best_curves.png', bbox_inches='tight')
    plt.show()
else:
    print('History best model tidak ada (model di-load dari cache). Re-run tanpa cache untuk curves.')

# 2. Loss curves dikelompokkan per variasi hyperparameter
if histories:
    hparam_groups = {
        'n_conv':  {'label': 'Jumlah Conv Layer', 'vals': sorted({r['n_conv']  for r in all_results})},
        'k_size':  {'label': 'Ukuran Filter',     'vals': sorted({r['k_size']  for r in all_results})},
        'pool':    {'label': 'Jenis Pooling',     'vals': sorted({r['pool']    for r in all_results})},
        'filters': {'label': 'Filter per Layer',  'vals': sorted({r['filters'] for r in all_results})},
    }
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    for param, info in hparam_groups.items():
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
        for ci, val in enumerate(info['vals']):
            subset = [r for r in all_results if str(r[param]) == str(val)]
            all_vl = [histories[r['tag']]['val_loss'] for r in subset if r['tag'] in histories]
            all_tl = [histories[r['tag']]['loss']     for r in subset if r['tag'] in histories]
            if not all_vl:
                continue
            max_e = max(len(v) for v in all_vl)
            pad = lambda arr: [v + [v[-1]] * (max_e - len(v)) for v in arr]
            mean_vl = np.array(pad(all_vl)).mean(axis=0)
            mean_tl = np.array(pad(all_tl)).mean(axis=0)
            c = colors[ci % len(colors)]
            ax1.plot(mean_tl, label=f'{param}={val}', color=c)
            ax2.plot(mean_vl, label=f'{param}={val}', color=c)
        ax1.set_title(f'Train Loss - {info["label"]}'); ax1.legend(); ax1.set_xlabel('Epoch')
        ax2.set_title(f'Val Loss - {info["label"]}');   ax2.legend(); ax2.set_xlabel('Epoch')
        plt.suptitle(f'Pengaruh {info["label"]} terhadap Loss')
        plt.tight_layout()
        plt.savefig(MODELS_DIR / f'loss_curves_{param}.png', bbox_inches='tight')
        plt.show()
else:
    print('Tidak ada history tersimpan. Re-run training tanpa cache untuk mendapatkan loss curves.')


## 5. Keras vs From-Scratch (Arsitektur Terbaik)

In [ ]:
from src.cnn.model import CNNFromScratch

best_path  = MODELS_DIR / f'arch_{best_row["idx"]:02d}_{best_tag}.h5'
keras_model = tf.keras.models.load_model(best_path)

# Convert ke numpy untuk from-scratch (load test set sekali)
X_test_np = np.concatenate([x.numpy() for x, _ in test_ds])
print(f'X_test shape: {X_test_np.shape}')

# Keras prediction
t0 = time.time()
y_keras = keras_model.predict(test_ds, verbose=0).argmax(1)
t_keras = time.time() - t0
f1_keras = f1_score(y_test, y_keras, average='macro')

# From-scratch prediction
scratch = CNNFromScratch.from_keras(keras_model)
t0 = time.time()
y_scratch = scratch.predict_classes(X_test_np)
t_scratch = time.time() - t0
f1_scratch = f1_score(y_test, y_scratch, average='macro')

print(f'\n{"":20s} {"F1-macro":>10s} {"Waktu (s)":>10s}')
print(f'{"Keras":20s} {f1_keras:>10.4f} {t_keras:>10.2f}')
print(f'{"From-Scratch":20s} {f1_scratch:>10.4f} {t_scratch:>10.2f}')
print(f'\nPrediksi identik: {np.all(y_keras == y_scratch)}')

In [ ]:
# Classification report lengkap
print('=== Keras ===')
print(classification_report(y_test, y_keras, target_names=CLASSES))
print('\n=== From-Scratch ===')
print(classification_report(y_test, y_scratch, target_names=CLASSES))

## 6. Conv2D vs LocallyConnected2D

> **Catatan:** LC2D pakai input size 32×32 (bukan 150×150) karena parameter-nya jauh lebih besar.
> Ini fair karena tujuannya membandingkan arsitektur, bukan ukuran input.
> LC2D pada 150×150 secara teori bisa dijalankan tapi butuh VRAM besar dan lama.

In [ ]:
IMG_SIZE_LC = 32   # input size lebih kecil buat LC2D

# Dataset versi 32x32 (buat LC2D)
train_lc = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=(IMG_SIZE_LC, IMG_SIZE_LC),
    batch_size=BATCH_SIZE, label_mode='int', seed=42, class_names=CLASSES,
).map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)

test_lc = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=(IMG_SIZE_LC, IMG_SIZE_LC),
    batch_size=BATCH_SIZE, label_mode='int', shuffle=False, class_names=CLASSES,
).map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)

y_test_lc = np.concatenate([y.numpy() for _, y in test_lc])

In [ ]:
def build_lc2d(n_conv, filters, k_size, pool_type, img_size=IMG_SIZE_LC):
    """Ganti Conv2D dengan LocallyConnected2D (no parameter sharing)."""
    PoolLayer = (tf.keras.layers.MaxPooling2D if pool_type == 'max'
                 else tf.keras.layers.AveragePooling2D)

    model = tf.keras.Sequential(name=f'lc2d_c{n_conv}_k{k_size}_{pool_type}')
    model.add(tf.keras.layers.Input(shape=(img_size, img_size, 3)))

    for i in range(n_conv):
        f = filters[i % len(filters)]
        # LC2D hanya support 'valid' padding
        model.add(tf.keras.layers.LocallyConnected2D(f, k_size, activation='relu'))
        model.add(PoolLayer(2, 2))

    model.add(tf.keras.layers.GlobalAveragePooling2D())
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'))

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Gunakan config terbaik dari Conv2D (ambil n_conv, filters, k_size, pool)
n_best  = best_row['n_conv']
f_best  = eval(best_row['filters'])  # string '[32, 64]' → list
k_best  = best_row['k_size']
p_best  = best_row['pool']

lc_model = build_lc2d(n_best, f_best, k_best, p_best)
lc_model.summary()

# Conv2D versi 32x32 buat perbandingan fair
conv_32 = build_cnn(n_best, f_best, k_best, p_best)
# Overwrite input size
conv_32 = tf.keras.Sequential(
    [tf.keras.layers.Input(shape=(IMG_SIZE_LC, IMG_SIZE_LC, 3))] +
    [l for l in conv_32.layers[1:]],
    name='conv2d_32x32'
)
conv_32.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print(f'\nParam Conv2D (32x32) : {conv_32.count_params():,}')
print(f'Param LC2D   (32x32) : {lc_model.count_params():,}')

In [ ]:
lc_save   = MODELS_DIR / f'lc2d_best.h5'
c32_save  = MODELS_DIR / f'conv2d_32x32_best.h5'

for mdl, path, ds_tr in [
    (conv_32, c32_save, train_lc),
    (lc_model, lc_save,  train_lc),
]:
    name = mdl.name
    if path.exists():
        print(f'{name}: loaded')
        mdl.set_weights(tf.keras.models.load_model(path).get_weights())
    else:
        print(f'Training {name} ...')
        h = mdl.fit(ds_tr, epochs=EPOCHS, validation_data=test_lc,
                    callbacks=[early_stop], verbose=0)
        mdl.save(path)
        histories[name] = h.history
        print(f'  Done ({len(h.history["loss"])} epochs)')

In [ ]:
# Muat ulang kalau dari cache
if lc_save.exists():
    lc_model = tf.keras.models.load_model(lc_save)
if c32_save.exists():
    conv_32 = tf.keras.models.load_model(c32_save)

y_conv32 = conv_32.predict(test_lc, verbose=0).argmax(1)
y_lc     = lc_model.predict(test_lc, verbose=0).argmax(1)

f1_conv32 = f1_score(y_test_lc, y_conv32, average='macro')
f1_lc     = f1_score(y_test_lc, y_lc,     average='macro')

print(f'\n{"":25s} {"F1-macro":>10s} {"Params":>12s}')
print(f'{"Conv2D (32x32)":25s} {f1_conv32:>10.4f} {conv_32.count_params():>12,}')
print(f'{"LC2D   (32x32)":25s} {f1_lc:>10.4f} {lc_model.count_params():>12,}')

In [ ]:
# Loss curves: Conv2D vs LC2D
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, h in [('conv2d_32x32', histories.get('conv2d_32x32')),
                ('lc2d', histories.get(lc_model.name))]:
    if h is None:
        continue
    axes[0].plot(h['loss'],      label=f'{name} train')
    axes[0].plot(h['val_loss'],  label=f'{name} val', linestyle='--')
    axes[1].plot(h['accuracy'],     label=f'{name} train')
    axes[1].plot(h['val_accuracy'], label=f'{name} val', linestyle='--')

axes[0].set_title('Loss'); axes[0].legend(fontsize=8)
axes[1].set_title('Accuracy'); axes[1].legend(fontsize=8)
plt.suptitle('Conv2D vs LocallyConnected2D')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'conv_vs_lc2d.png', bbox_inches='tight')
plt.show()
print('\nAnalisis: LC2D punya lebih banyak parameter (tidak ada parameter sharing),')
print('tapi belum tentu F1-nya lebih baik — tergantung kapasitas data vs model size.')